<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/RESUME_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q pypdf

In [16]:
from google.colab import files
uploaded = files.upload()

Saving Naga_Gopal_Resume.pdf to Naga_Gopal_Resume (1).pdf


In [17]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

# Extract all text
full_text = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        full_text += text + "\n"

print("Total characters:", len(full_text))


Total characters: 4391


In [18]:
chunk_size = 1000
overlap = 200

chunks = []

start = 0

while start < len(full_text):
    end = start + chunk_size

    chunk = full_text[start:end]

    chunks.append(chunk)

    start += chunk_size - overlap

print("Number of chunks:", len(chunks))

Number of chunks: 6


In [19]:
!pip install -q sentence-transformers

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
embeddings = model.encode(chunks)

print(type(embeddings))
print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)

<class 'numpy.ndarray'>
Number of embeddings: 6
Embedding shape: (6, 384)


In [22]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU: Tesla T4


In [23]:
!pip install -q faiss-cpu
import faiss

print("FAISS version:", faiss.__version__)
print("Number of GPUs detected by FAISS:", faiss.get_num_gpus())

FAISS version: 1.15.1
Number of GPUs detected by FAISS: 0


In [24]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Index type:", type(index))
print("Vectors stored:", index.ntotal)

Index type: <class 'faiss.swigfaiss.IndexFlatL2'>
Vectors stored: 6


In [25]:
!pip install -q google-genai

In [26]:
from google.colab import userdata
api_key=userdata.get('api_key')

In [28]:
from google import genai

client = genai.Client(api_key=api_key)

In [30]:
query = "What are the skills mentioned in the resume?"
import numpy as np
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

D, I = index.search(query_embedding, 5)

context = "\n\n".join([chunks[i] for i in I[0]])

prompt = f"""You are a helpful assistant.
Answer ONLY using the context below.
If answer not found say "not in document."

Context:
{context}

Question: {query}

Answer:"""

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=prompt
)
print(response.text)


Based on the provided context, the skills mentioned are:

* **GEN AI / LLMS:** AI Agents, RAG, NLP, Transformers
* **ML / DL:** CNN, LSTM, ANN, RNN, Vision Transformer (ViT), Attention Mechanisms, LoRA Fine-Tuning
* **GENERATIVE:** Diffusion-Based Image Generation, Open-Source Video Generation, Background Removal, Image Upscaling
* **CV & MEDIA:** OpenCV, YOLO, MediaPipe, FFmpeg, Image Processing, NumPy, Pandas, Matplotlib
* **BACKEND:** Python, Node.js, Express.js, REST APIs, FastAPI, Async Job Processing, SQL, MySQL
* **FRONTEND & TOOLS:** React.js (Basics), JavaScript, HTML, CSS, Bootstrap, Git, GitHub, Postman, Jupyter Notebook
